# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
*Title*: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells  
*Description*: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.  
*Croissant schema URL*: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata  # This is an object with fields such as name, description, etc.
print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review the available record sets, their fields, and associated IDs. All entities are referenced by their `@id`s as per the Croissant specification.

In [ ]:
# List available record sets and associated fields/columns via their @id
print("Available record sets and their fields/columns:\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, type: {getattr(field, 'data_type', 'unknown')})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

We select the first record set for detailed analysis below.

In [ ]:
# Extract and load data for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs in dataset.record_sets:
    print(f"Loading records for record set {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found for this record set.")

# For the rest of the notebook, use the first available record set with data
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
We perform common data processing steps: filtering records above a threshold in a numeric field, normalizing, and grouping. All field references use their `@id` as required.

_First, identify numeric fields to use by field `@id`:_

In [ ]:
# Identify a suitable numeric field for filtering and normalization
numeric_candidates = []
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            dt = getattr(field, 'data_type', None)
            if dt in ['Float', 'Number', 'Integer']:
                numeric_candidates.append(field.id)
if not numeric_candidates:
    raise ValueError('No numeric fields found in the selected record set.')
print(f"Numeric candidate fields (@id): {numeric_candidates}")
# We'll take the first numeric field for demonstration
numeric_field_id = numeric_candidates[0]
print(f"Using field @id: {numeric_field_id}")

In [ ]:
# EDA steps using the chosen numeric field
df = dataframes[main_record_set_id]
threshold = df[numeric_field_id].quantile(0.90) if df[numeric_field_id].dtype.kind in 'fi' else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (top 5 shown):")
print(filtered_df.head())

# Normalization
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (top 5 shown):")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by a non-numeric field (if any present)
group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'O']
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(grouped_df.head())
else:
    print("No suitable non-numeric field for grouping found.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and relationships between fields. This step uses the `@id` for all variable references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping field exists, bar plot of group means
if 'group_field_id' in locals():
    plt.figure(figsize=(10,4))
    ax = sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
- We loaded, explored, and processed the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`.
- We reviewed available record sets, fields (all referenced by `@id`), and analyzed their data in a pandas DataFrame.
- Exploratory data analysis included filtering by a numeric field, normalization, and visual exploration.
- All steps dynamically referenced dataset elements by `@id`, facilitating robust and schema-consistent data processing.

Further analysis may include domain-specific feature engineering, downstream machine learning, or cross-dataset integration using the Croissant schema definitions.